# Evaluate Champion Model
Load current champion from registry and compare with new candidates

In [ ]:
%pip install --quiet mlflow==3.6.0 tensorflow==2.20.0 xgboost==3.1.3 scikit-learn==1.8.0
dbutils.library.restartPython()

In [ ]:
import numpy as np
import os
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.tensorflow
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.set_registry_uri('databricks-uc')

# Get parameters from DAB
try:
    volume_path = dbutils.widgets.get('volume_path')
    catalog_name = dbutils.widgets.get('catalog_name')
    models_schema = dbutils.widgets.get('models_schema')
    model_name = dbutils.widgets.get('model_name')
except:
    volume_path = '/Volumes/ai_ml_in_practice/mnist_data/processed'
    catalog_name = 'ai_ml_in_practice'
    models_schema = 'dab_models'
    model_name = 'mnist_digit_classifier'

print(f'Volume path: {volume_path}')
print(f'Full model name: {catalog_name}.{models_schema}.{model_name}\n')

# Load test data
x_test_flat = np.load(os.path.join(volume_path, 'x_test.npy'))
x_test_images = np.load(os.path.join(volume_path, 'x_test_images.npy'))
y_test = np.load(os.path.join(volume_path, 'y_test.npy'))
y_test_images = np.load(os.path.join(volume_path, 'y_test_images.npy'))

x_test_images = x_test_images[..., np.newaxis]

print(f'Test data - flat: {x_test_flat.shape}, images: {x_test_images.shape}')

min_accuracy = 0.90
min_f1 = 0.90
print(f'Quality gates: accuracy >= {min_accuracy}, F1 >= {min_f1}\n')

In [ ]:
# Try to load champion model from registry
full_model_name = f'{catalog_name}.{models_schema}.{model_name}'
champion_run_id = None
champion_metrics = None

try:
    # Try to get champion version
    client = mlflow.tracking.MlflowClient()
    aliases = client.get_model_version_by_alias(full_model_name, 'champion')
    champion_version = aliases.version
    
    # Load champion model
    champion_uri = f'models:/{full_model_name}@champion'
    print(f'Found champion model at version {champion_version}')
    
    # Get metadata to determine algorithm
    model_version = client.get_model_version(full_model_name, champion_version)
    algorithm = model_version.tags.get('algorithm', 'unknown')
    print(f'Algorithm: {algorithm}')
    
    # Load and evaluate based on algorithm
    if algorithm == 'logistic':
        model = mlflow.sklearn.load_model(champion_uri)
        y_pred = model.predict(x_test_flat)
        y_proba = model.predict_proba(x_test_flat)
        test_data = x_test_flat
    elif algorithm == 'xgboost':
        model = mlflow.xgboost.load_model(champion_uri)
        y_pred = model.predict(x_test_flat)
        y_proba = model.predict_proba(x_test_flat)
        test_data = x_test_flat
    elif algorithm == 'cnn':
        model = mlflow.tensorflow.load_model(champion_uri)
        y_proba = model.predict(x_test_images, verbose=0)
        y_pred = np.argmax(y_proba, axis=1)
        test_data = x_test_images
        y_test = y_test_images
    else:
        raise RuntimeError(f'Unknown champion algorithm: {algorithm}')
    
    # Evaluate champion
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
    
    champion_metrics = {
        'model': algorithm,
        'version': champion_version,
        'accuracy': acc,
        'f1': f1,
        'auc': auc
    }
    
    print(f'  Accuracy: {acc:.4f}, F1: {f1:.4f}\n')
    
except Exception as e:
    print(f'No champion model found (first run): {e}')
    print('Will compare only new candidates\n')

In [ ]:
# Pass champion info to next task
try:
    if champion_metrics:
        dbutils.jobs.taskValues.set(key='champion_accuracy', value=str(champion_metrics['accuracy']))
        dbutils.jobs.taskValues.set(key='champion_f1', value=str(champion_metrics['f1']))
        dbutils.jobs.taskValues.set(key='champion_algorithm', value=champion_metrics['model'])
        dbutils.jobs.taskValues.set(key='champion_version', value=str(champion_metrics['version']))
        dbutils.jobs.taskValues.set(key='champion_exists', value='True')
        print('Champion metrics passed to register_model task')
    else:
        dbutils.jobs.taskValues.set(key='champion_exists', value='False')
        print('No champion to compare - will register best candidate')
except Exception as e:
    logger.warning(f'Could not set task values: {e}')